In [2]:
import networkx as nx
from bp.world import random_grid_world, Scenario

In [3]:
world = random_grid_world(
    rows=4,
    cols=4,
    demands={
        # ((0, 0), (3, 3)): 10,
        # ((1, 0), (1, 3)): 5,
        ((1, 0), (1, 3)): 6,
        ((0, 0), (2, 3)): 6,
    },
    seed=0,
)
G = world.network.graph
world.network.bpr_beta = 1.0
nominal_ = Scenario.from_world("nominal", world)
# a bit hacky, but lazy and want to avoid errors

accident_congestion = 10
target_edge = ((1, 1), (1, 2))

G.edges[target_edge]["travel_time"] += accident_congestion
travel_time = dict(nominal_.travel_time)
travel_time[target_edge] += accident_congestion
nominal = Scenario(
    name="accident",
    travel_time=travel_time,
    discomfort=nominal_.discomfort,
    hazard=nominal_.hazard,
    cost=nominal_.cost,
    emissions=nominal_.emissions,
    policing=nominal_.policing
)

print(f"{world.total_population=}")
# print(f"{world.total_demand((0, 0), (3, 3))=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")

world.total_population=12
world.total_demand((1, 0), (1, 3))=6


In [4]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("travel time")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
t = world.network.travel_time
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

Set parameter Username
Set parameter LicenseID to value 2844113
Academic license - for non-commercial use only - expires 2027-07-13


In [5]:
import numpy as np

def debug_scenario(volumes):
    # (volume - 1) b/c need to recalculate b/c of off-by-one-error
    active_travel_time = world.network.actual_travel_times({a: volume - 1 for a, volume in volumes.items()})
    active_arcs = {a: t for a, t in active_travel_time.items() if volumes[a] > 0}

    travel_time = sum(t * volumes[a] for a, t in active_arcs.items())
    average_travel_time = travel_time / world.total_population
    print(f"{travel_time=:.2f}")
    print(f"{average_travel_time=:.2f}")

    nominal_arc_time = sum(nominal.travel_time[a] for a in active_arcs)
    realized_arc_time = sum(active_arcs.values())
    average_arc_time = realized_arc_time / len(active_arcs)
    arc_time_increase = realized_arc_time - nominal_arc_time
    average_arc_time_increase = arc_time_increase / len(active_arcs)
    print(f"{average_arc_time=:.2f} (along arcs w/ non-zero flow/volume)")
    print(f"{average_arc_time_increase=:.2f} (+{arc_time_increase / nominal_arc_time * 100:.2f}%, overall, not per-arc)")

    flow = np.array([volumes.get(arc, 0) for arc in world.ordered_arcs], dtype=float)
    capacity_used = flow / world.network._arc_array("capacity") * 100
    active_capacity_used = capacity_used[capacity_used > 0]
    average_capacity_used = np.average(active_capacity_used)
    most_capacity_used = np.max(capacity_used)
    print(f"{most_capacity_used=:.2f}%")
    print(f"{average_capacity_used=:.2f}%")

    # should sort then bin-search for `congested` b/c sorting but copy paste lazy
    congested = sum(active_arcs[a] > nominal.travel_time[a] + 1e-9 for a in active_arcs)
    print(f"arcs whose travel time rose under congestion: {congested}/{len(world.A)}")

    most_congested = sorted(((a, (active_arcs[a] - nominal.travel_time[a]) / nominal.travel_time[a]) for a in active_arcs), reverse=True, key=lambda x: x[1])
    for i, ((orig, dest), delta) in enumerate(most_congested[:congested], 1):
        print(f"{i}: {orig}->{dest}: +{delta * 100:.2f}%")

    return travel_time

In [6]:
# TODO: investigate using numpy for precomputing `tau`
# world.network.actual_travel_times()

# b/c of off-by-one-error, we modify tau definition
# NOTE: for x=0, value is inaccurate, but irrelevant b/c flow is also zero
# original: tau[a, x] := average cost for (x + 1) players on arc a
# modified: tau[a, x] := average cost for x players on arc a
tau = {}
for a in A:
    for x in range(n + 1):
        tau[a, x] = t[a] * (1 + alpha * ((x - 1) / c[a]) ** beta)

# decision: r[i, a] := 1 iff player i uses arc a
# r = model.addVars(N, A, vtype=GRB.BINARY, name="r")
r = {
    (i, a): model.addVar(vtype=GRB.BINARY, name=f"r_{i.id}_{a}")
    for i in N for a in A
}

# decision: x[a] := total flow on arc a
x = {
    a: model.addVar(vtype=GRB.INTEGER, lb=0, ub=n, name=f"x_{a}")
    for a in A
}

# constraint: flow conservation
for i in N:
    for v in V:
        flow_out = gp.quicksum(r[i, a] for a in A if a[0] == v)
        flow_in = gp.quicksum(r[i, a] for a in A if a[1] == v)
        flow = 1 if v == i.demand.origin else (-1 if v == i.demand.destination else 0)

        model.addConstr(flow_out - flow_in == flow, name=f"flow_{i.id}_{v}")

# constraint: x[a] = \sum_{i \in N} r[i, a]
for a in A:
    model.addConstr(
        x[a] == gp.quicksum(r[i, a] for i in N),
        name=f"x_{a}"
    )

In [7]:
def eval_model():
    model.optimize()
    assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

    # routes = {
    #     i: [a for a in A if r[i, a].X]
    #     for i in N
    # }

    # volumes = {a: 0 for a in A}
    # for route in routes.values():
    #     for a in route:
    #         volumes[a] += 1

    # return debug_scenario(world.realize(routes), omega, volumes)

    return debug_scenario({
        a: sum(1 for i in N if r[i, a].X)
        for a in A
    })

In [8]:
# objective: \min \sum_{a \in A} x[a] \cdot tau[a, x[a]]
for a in A:
    model.setPWLObj(
        x[a],
        list(range(n + 1)),
        [k * tau[a, k] for k in range(n + 1)]
    )

print("OPTIMAL:")
travel_time_optimal = eval_model()
assert abs(model.ObjVal - travel_time_optimal) < 1e-5, "obj value and realized travel time mismatch"

OPTIMAL:
travel_time=384.88
average_travel_time=32.07
average_arc_time=6.95 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=0.52 (+8.11%, overall, not per-arc)
most_capacity_used=249.83%
average_capacity_used=95.73%
arcs whose travel time rose under congestion: 14/48
1: (0, 3)->(1, 3): +28.11%
2: (0, 1)->(0, 2): +22.30%
3: (1, 0)->(1, 1): +18.87%
4: (2, 2)->(2, 3): +17.86%
5: (2, 1)->(2, 2): +17.65%
6: (0, 0)->(0, 1): +14.43%
7: (0, 0)->(1, 0): +11.49%
8: (0, 2)->(0, 3): +9.87%
9: (1, 2)->(1, 3): +9.83%
10: (1, 1)->(2, 1): +9.20%
11: (2, 0)->(2, 1): +6.10%
12: (1, 1)->(0, 1): +5.28%
13: (2, 2)->(1, 2): +3.23%
14: (1, 0)->(2, 0): +3.11%


In [9]:
# objective: \min \sum_{a \in A} x[a] \cdot tau[a, x[a]]

# objective: \min \sum_{a \in A} \sum_{k=1}^{x[a]} tau[a, k]

for a in A:
    model.setPWLObj(
        x[a],
        list(range(n + 1)),
        [(sum(tau[a, k] for k in range(1, x + 1))) for x in range(n + 1)]
    )

print("SELFISH:")
travel_time_selfish = eval_model()
# don't assert b/c the obj. is different

SELFISH:
travel_time=385.89
average_travel_time=32.16
average_arc_time=6.33 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=0.61 (+10.75%, overall, not per-arc)
most_capacity_used=312.28%
average_capacity_used=107.43%
arcs whose travel time rose under congestion: 13/48
1: (0, 3)->(1, 3): +37.47%
2: (0, 1)->(0, 2): +27.87%
3: (0, 0)->(0, 1): +21.64%
4: (1, 0)->(1, 1): +18.87%
5: (2, 1)->(2, 2): +17.65%
6: (2, 2)->(2, 3): +13.39%
7: (0, 2)->(0, 3): +13.16%
8: (1, 1)->(2, 1): +12.27%
9: (1, 2)->(1, 3): +9.83%
10: (0, 0)->(1, 0): +7.66%
11: (2, 2)->(1, 2): +6.46%
12: (1, 3)->(2, 3): +6.05%
13: (1, 1)->(0, 1): +5.28%


In [10]:
# decision: z[a, k] := 1[x[a] = k] (one-hot encoding of arc flow)
z = {
    (a, k): model.addVar(vtype=GRB.BINARY, name=f"z_{a}_{k}")
    for a in A for k in range(n + 1)
}

# constraint: z[a, k] = 1[x[a] = k]
for a in A:
    model.addConstr(gp.quicksum(z[a, k] for k in range(n + 1)) == 1, name=f"one-hot_{a}")
    model.addConstr(gp.quicksum(k * z[a, k] for k in range(n + 1)) == x[a], name=f"z_{a}")

# objective: \min \sum_{a \in A} \sum_{k=0}^{n} k \cdot tau[a, k] \cdot z[a, k]
model.setObjective(
    gp.quicksum(
        (k * tau[a, k]) * z[a, k]
        for a in A for k in range(n + 1)
    ),
    GRB.MINIMIZE
)

print("OPTIMAL (one-hot):")
travel_time_optimal_one_hot = eval_model()
assert abs(model.ObjVal - travel_time_optimal_one_hot) < 1e-5, "obj value and realized travel time mismatch"

OPTIMAL (one-hot):
travel_time=384.88
average_travel_time=32.07
average_arc_time=6.95 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=0.52 (+8.11%, overall, not per-arc)
most_capacity_used=249.83%
average_capacity_used=95.73%
arcs whose travel time rose under congestion: 14/48
1: (0, 3)->(1, 3): +28.11%
2: (0, 1)->(0, 2): +22.30%
3: (1, 0)->(1, 1): +18.87%
4: (2, 2)->(2, 3): +17.86%
5: (2, 1)->(2, 2): +17.65%
6: (0, 0)->(0, 1): +14.43%
7: (0, 0)->(1, 0): +11.49%
8: (0, 2)->(0, 3): +9.87%
9: (1, 2)->(1, 3): +9.83%
10: (1, 1)->(2, 1): +9.20%
11: (2, 0)->(2, 1): +6.10%
12: (1, 1)->(0, 1): +5.28%
13: (2, 2)->(1, 2): +3.23%
14: (1, 0)->(2, 0): +3.11%


In [11]:
# lol just making stuff up - am i misrepresenting PoA
price_anarchy = (travel_time_selfish - travel_time_optimal) / travel_time_optimal * 100
print(f"{price_anarchy=:.2f}%")

assert travel_time_optimal == travel_time_optimal_one_hot, "one-hot implem deviates"

price_anarchy=0.26%
